# 01 - Loading the data and building a baseline
BCS 3101 Capstone: predicting telecom customer churn

**What this notebook does**
1. Load the data and look at it.
2. Fix one column that has the wrong data type.
3. Split the data into a training part and a testing part.
4. Build a "preprocessing pipeline" - a set of steps that cleans and prepares the data for a model.
5. Set a baseline - the simplest possible guess - so we know what any real model has to beat.



In [ ]:
# --- Import the libraries---
# pandas: for working with tables of data (like Excel, but in code)
# numpy: for numeric calculations
# matplotlib / seaborn: for making charts
# sklearn (scikit-learn): the machine learning library

import warnings
warnings.filterwarnings('ignore')  # hide harmless warning messages so the output stays readable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')  # makes charts look a bit nicer

print('Libraries loaded.')

Libraries loaded.


## Step 1: Load the data and look at it
We read the CSV file into a table called a "DataFrame". `df` is just a short name for it

In [2]:
df = pd.read_csv('../data/raw/telco.csv')

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

df.head()   # shows the first 5 rows, so we can see what the data looks like

Rows: 7043
Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# .info() tells us the column names, how many values each has, and what data TYPE each column is
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

Every column from the output above should be a sensible type:
- numbers should say `int64` or `float64`
- text/categories should say `object`

One column does NOT look right: **TotalCharges** should be numbers (it's money), but it says `object`,
which means pandas is treating it as text. Let's find out why.

In [4]:
# Let's look for rows where TotalCharges is not a normal number.
# .str.strip() removes extra spaces. If what's left is an empty string '', that's our problem row.

is_blank = df['TotalCharges'].str.strip() == ''
print('Number of blank TotalCharges values:', is_blank.sum())

# show those exact rows so we can see what they have in common
df.loc[is_blank, ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

Number of blank TotalCharges values: 11


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,,No
753,0,20.25,,No
936,0,80.85,,No
1082,0,25.75,,No
1340,0,56.05,,No
3331,0,19.85,,No
3826,0,25.35,,No
4380,0,20.00,,No
5218,0,19.70,,No
6670,0,73.35,,No




All the 11 customers have a tenure of 0, meaning they are new customers who have not yet completed a month of service. This could explain why their TotalCharges values are blank, since they may not have accumulated any charges yet. However, they already have monthly charges, which suggests that they have subscribed to a service but have not yet accumulated a total billing amount.

## Step 2: Fix the data types
Now we clean up the data before we do anything else with it. Two small fixes:

1. Turn `TotalCharges` into real numbers. The blank values become `NaN` ("Not a Number"), which is
   pandas' way of marking a missing value. We do NOT fill them in yet - that happens later, and only
   using the training data, so we don't cheat.
2. Turn the target column `Churn` (currently "Yes"/"No") into 1/0, because machine learning models
   need numbers, not words, for the thing they are predicting.

In [5]:
# pd.to_numeric turns text into numbers. errors='coerce' means: if something can't be
# turned into a number, replace it with NaN instead of crashing.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('Missing values in TotalCharges now:', df['TotalCharges'].isna().sum())

# customerID is just a random ID string, like a serial number - it can't help predict anything,
# so we remove it.
df = df.drop(columns=['customerID'])

# Turn the target into 1 = churned, 0 = did not churn
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

df.head()

Missing values in TotalCharges now: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Step 3: Split into training data and testing data
We train the model on one part of the data (the **training set**) and check how well it works on
data it has never seen (the **test set**). This is how we find out if a model actually learned
something useful, instead of just memorising the training data.

We use `stratify=y` so that both the training set and the test set have the same percentage of
churned customers as the full dataset. Without this, one split could end up with way more (or fewer)
churners than the other by chance.

In [6]:
from sklearn.model_selection import train_test_split

# X = the columns we use to make a prediction (everything except Churn)
# y = the thing we are trying to predict (the Churn column)
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% of the data goes into the test set, 80% into training
    stratify=y,           # keep the same churn percentage in both sets
    random_state=42,      # a fixed number so we get the SAME split every time we run this cell
)

print('Training rows:', X_train.shape[0])
print('Testing rows: ', X_test.shape[0])
print()
print('Churn rate in full data:    ', round(y.mean(), 3))
print('Churn rate in training set:', round(y_train.mean(), 3))
print('Churn rate in test set:    ', round(y_test.mean(), 3))

Training rows: 5634
Testing rows:  1409

Churn rate in full data:     0.265
Churn rate in training set: 0.265
Churn rate in test set:     0.265


## Step 4: Build the preprocessing pipeline
Before a model can use this data, we need to:

- **fill in missing numbers** (like the blank TotalCharges values) - we use the **median**, since it
  isn't thrown off by a few unusually high or low values
- **scale the numbers** so they're all on a similar range (a model can get confused if `tenure` goes
  0-72 but `TotalCharges` goes 0-8000)
- **turn categories into numbers** - a model can't read the word "Yes" or "DSL", so we turn each
  category into its own 0/1 column. This is called **one-hot encoding**.

Doing all of this by hand, column by column, would be a lot of repeated code. Scikit-learn gives us
two tools that do it for us and - importantly - do it the SAFE way:

- **`Pipeline`**: chains steps together (fill missing → scale) so they always run in the right order.
- **`ColumnTransformer`**: applies different treatment to different columns (numbers get one treatment,
  categories get another) inside a single step.

**Why "safe"?** If we filled in missing values or scaled the numbers using the WHOLE dataset before
splitting, information from the test set would leak into training - the model would get a sneak peek
at data it's supposed to be tested on. Using `Pipeline`/`ColumnTransformer` and only calling `.fit()`
on the training set stops that from happening.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# List the numeric columns and the category (text) columns separately,
# because they each need different treatment.

numeric_columns = ['tenure', 'MonthlyCharges', 'TotalCharges']

category_columns = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
]

# Steps for the numeric columns: first fill in missing values, then scale them
numeric_steps = Pipeline([
    ('fill_missing', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

# Step for the category columns: turn each category into its own 0/1 column
category_steps = OneHotEncoder(handle_unknown='ignore')
# handle_unknown='ignore' means: if the test set has a category the training set never saw,
# don't crash - just treat it as all zeros.

# Now combine both: apply numeric_steps to the numeric columns,
# and category_steps to the category columns
preprocessing = ColumnTransformer([
    ('numbers', numeric_steps, numeric_columns),
    ('categories', category_steps, category_columns),
])

print('Pipeline built.')

Pipeline built.


**Proof this is leakage-safe:** we `.fit()` the pipeline on the TRAINING data only. If we then check
the average value it learned for each numeric column, it should match the training set's average -
NOT the average of the whole dataset.

In [8]:
preprocessing.fit(X_train)   # learn everything (medians, scaling, categories) from TRAINING data only

# dig out the scaler to check what average value it learned
scaler = preprocessing.named_transformers_['numbers'].named_steps['scale']

check = pd.DataFrame({
    'learned by the pipeline': scaler.mean_,
    'training set average':    X_train[numeric_columns].mean().values,
    'FULL dataset average':    df[numeric_columns].mean().values,
}, index=numeric_columns)

check.round(2)

,learned by the pipeline,training set average,FULL dataset average
tenure,32.49,32.49,32.37
MonthlyCharges,64.93,64.93,64.76
TotalCharges,2301.32,2302.60,2283.30


From the table above: the "learned by the pipeline" column matches the "training set average"
column almost exactly, and is slightly different from the "FULL dataset average" column. That
difference is small here, but it is proof the pipeline only looked at training data - exactly what
we want.

In [9]:
# quick check: after preprocessing, is there any missing data left, and how many columns do we have now?
X_train_ready = preprocessing.transform(X_train)

print('Shape after preprocessing:', X_train_ready.shape)
print('Any missing values left:', np.isnan(X_train_ready).any())

Shape after preprocessing: (5634, 46)
Any missing values left: False


## Step 5: Set the baseline
Before trying any real model, we check the simplest possible "model": just guess the most common
answer every time ("this customer will NOT churn"). Any real model we build later MUST beat this,
or it isn't worth using.

We check three scores, not just one:
- **accuracy**: percent of correct guesses (can be misleading when one class is much bigger)
- **F1 score**: balances catching churners against not raising too many false alarms
- **ROC-AUC**: how well the model can tell churners and non-churners apart, regardless of threshold

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Put the "always guess no churn" rule together with our pipeline, so it's tested the same fair way
baseline_model = Pipeline([
    ('preprocessing', preprocessing),
    ('model', DummyClassifier(strategy='most_frequent')),
])

# 5-fold cross-validation: split the training data into 5 parts, train on 4, test on 1,
# repeat 5 times so every part gets tested once. This gives a more reliable score than
# just one train/test split.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = cross_validate(
    baseline_model, X_train, y_train, cv=cv,
    scoring=['accuracy', 'f1', 'roc_auc'],
)

print('Baseline ("always guess no churn") scores across 5 folds:')
print('accuracy:', results['test_accuracy'].round(3))
print('f1:      ', results['test_f1'].round(3))
print('roc_auc: ', results['test_roc_auc'].round(3))
print()
print('accuracy average:', round(results['test_accuracy'].mean(), 3))
print('f1 average:      ', round(results['test_f1'].mean(), 3))
print('roc_auc average: ', round(results['test_roc_auc'].mean(), 3))

Baseline ("always guess no churn") scores across 5 folds:
accuracy: [0.735 0.735 0.735 0.735 0.734]
f1:       [0. 0. 0. 0. 0.]
roc_auc:  [0.5 0.5 0.5 0.5 0.5]

accuracy average: 0.735
f1 average:       0.0
roc_auc average:  0.5


The baseline has a high accuracy because most customers in the dataset did not churn, so always predicting that a customer will stay gives many correct predictions. However, it fails to identify any customers who actually churned, which is why its F1 score is 0. The ROC-AUC is 0.5 because the model cannot distinguish between customers who churn and those who do not. This shows that accuracy alone can be misleading, since a model may appear to perform well while failing to identify the customers we are actually interested in.

